# Photometry Mixed-Effects Model 

Imports .csv generated from "Headfixed photometry generating linear model dataframe from all sessions" code.

Options to generate the model based on the lick, post-lick, or combined epoch.

Predictors: **restriction**, **session**, **trial**, **solution**, **previous solution**, **lick count** (centered), with **mouse x session** variance as a random effect. Optional interactions: `LICK_X_RESTRICTION`, `LICK_X_SOLUTION`, and `RESTRICTION_X_SOLUTION`.

This code was generated with assistance from Cursor AI coding software.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# --- User config for combined.csv structure ---
input_csv = './Output dataframes/combined.csv'
output_dir_root = Path('./Linear model output')

# Which deprivation trials to include in the model:
# "both" | "water restricted" | "food restricted"
restriction_filter = "both"

# Optional session filter:
# - None: include all sessions
# - (start, end): inclusive session range, e.g. (3, 5)
# - [s1, s2, ...]: explicit session numbers, e.g. [1, 3, 5]
session_filter = None

# Optional: allow lick-count slope to differ by restriction state and/or solution.
# lick×restriction: only when both restriction levels are modeled (restriction_filter == "both").
LICK_X_RESTRICTION = False
LICK_X_SOLUTION = False

# Optional: restriction×solution interaction (allows restriction effect to differ by offered solution).
# Only applied when restriction_filter == "both" (two restriction levels in the model).
RESTRICTION_X_SOLUTION = False

# How to treat prev_solution: 
# "treatment" = compare to reference level (none) | "sum" = compare to mean of all levels
CATEGORICAL_CONTRAST = "sum"  
_cc_valid = {"treatment", "sum"}
if CATEGORICAL_CONTRAST not in _cc_valid:
    raise ValueError(f"CATEGORICAL_CONTRAST must be one of {_cc_valid}, got {CATEGORICAL_CONTRAST!r}")

# --- Epoch & photometry column ---
# Which epoch to model
# "lick" | "post_lick" | "lick_and_post_licking"
EPOCH = "lick_and_post_licking"

# Photometry column read from the CSV for every row (same column for all epochs).
signal_col = "basesnip_z avg"  # or "dFF avg" 

# Raw strings in CSV `period` column -> canonical epoch keys (edit keys to match your file).
period_map = {
    "base": "base",
    "baseline": "base",
    "lick": "lick",
    "licking": "lick",
    "post_lick": "post_lick",
    "post_licking": "post_lick",
    "lick_and_post_licking": "lick_and_post_licking",
}

_valid_epochs = {"base", "lick", "post_lick", "lick_and_post_licking"}
if EPOCH not in _valid_epochs:
    raise ValueError(f"EPOCH must be one of {_valid_epochs}, got {EPOCH!r}")
if EPOCH not in set(period_map.values()):
    raise ValueError(
        f"EPOCH={EPOCH!r} must appear as a value in period_map (at least one raw label must map to it)."
    )

_run_tag = {"both": "both", "water restricted": "water_only", "food restricted": "food_only"}[restriction_filter]
if session_filter is None:
    _session_tag = "all_sessions"
elif isinstance(session_filter, tuple) and len(session_filter) == 2:
    _session_tag = f"sessions_{int(session_filter[0])}_to_{int(session_filter[1])}"
else:
    _session_tag = "sessions_" + "_".join(str(int(s)) for s in session_filter)
_epoch_slug = EPOCH.replace(" ", "_")
_model_tag = (
    f"epoch_{_epoch_slug}"
    + ("__sumPrev" if CATEGORICAL_CONTRAST == "sum" else "")
    + ("__lxR" if LICK_X_RESTRICTION else "")
    + ("__lxS" if LICK_X_SOLUTION else "")
    + ("__rxS" if RESTRICTION_X_SOLUTION else "")
)
output_dir = output_dir_root / _run_tag / _session_tag / _model_tag
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {output_dir}")
print(f"Modeling epoch: {EPOCH!r} (column {signal_col!r})")

# Code maps (update if your coding changes).
restriction_map = {
    1: "water restricted",
    2: "food restricted",
}

solution_map = {
    1: "dry",
    2: "water",
    3: "10% sucrose",
    4: "20% sucrose",
    5: "saccharine",
}

In [ ]:
df = pd.read_csv(input_csv)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
print(df.columns.tolist())

# Keep required columns
work = df[[
    "mouse_id", "day", "trial_number", "restriction_state", "spout_code",
    "previous_trial", "total_licks", "period", signal_col
]].copy()

work = work.rename(columns={
    "mouse_id": "mouse",
    "day": "session",
    "trial_number": "trial",
    "restriction_state": "restriction",
    "spout_code": "spout_code",
    "previous_trial": "prev_spout_code",
    "total_licks": "lick_count",
    "period": "period",
    signal_col: "signal",
})

# Coerce types.
work["mouse"] = work["mouse"].astype(str)
work["session"] = pd.to_numeric(work["session"], errors="coerce")
work["trial"] = pd.to_numeric(work["trial"], errors="coerce")
work["restriction"] = pd.to_numeric(work["restriction"], errors="coerce")
work["spout_code"] = pd.to_numeric(work["spout_code"], errors="coerce")
work["prev_spout_code"] = pd.to_numeric(work["prev_spout_code"], errors="coerce")
work["lick_count"] = pd.to_numeric(work["lick_count"], errors="coerce")
work["period"] = work["period"].astype(str)
work["signal"] = pd.to_numeric(work["signal"], errors="coerce")

# Map period labels and keep one row per trial for the selected epoch.
work["epoch"] = work["period"].map(period_map)
_n_unknown = work["epoch"].isna().sum()
if _n_unknown:
    print(f"Note: {_n_unknown} rows have unknown `period` labels (not in period_map); excluded below.")
unknown_periods = work.loc[work["epoch"].isna(), "period"].dropna().unique()
if len(unknown_periods):
    print(f"Unknown period values (sample): {unknown_periods[:20].tolist()}")

long_df = work[work["epoch"] == EPOCH].copy()
long_df = long_df.rename(columns={"spout_code": "solution_code"})

if long_df.empty:
    raise ValueError(
        f"No rows left for EPOCH={EPOCH!r}. Check period_map and CSV `period` values. "
        f"Canonical epochs present in data: {sorted(work['epoch'].dropna().unique().tolist())}"
    )

print(f"Using epoch {EPOCH!r}: {len(long_df):,} trial rows")

# Decode categorical labels.
long_df["restriction"] = long_df["restriction"].map(restriction_map).fillna(long_df["restriction"].astype(str))
long_df["solution"] = long_df["solution_code"].map(solution_map)
long_df["prev_solution"] = long_df["prev_spout_code"].map(solution_map)
long_df["prev_solution"] = long_df["prev_solution"].fillna("none")

# Drop rows with missing required fields.
required = ["mouse", "session", "trial", "signal", "restriction", "solution", "prev_solution", "lick_count"]
long_df = long_df.dropna(subset=required).reset_index(drop=True)

print(long_df.head())
print(f"Rows (one per trial, {EPOCH} signal): {len(long_df):,}")

In [ ]:
# Optional: restrict to one deprivation state (before categoricals).
_valid_rf = {"both", "water restricted", "food restricted"}
if restriction_filter not in _valid_rf:
    raise ValueError(f"restriction_filter must be one of {_valid_rf}")
if restriction_filter != "both":
    long_df = long_df[long_df["restriction"] == restriction_filter].copy()
    print(f"Filtered to {restriction_filter!r}: {len(long_df)} rows")

# Optional: restrict to selected sessions (before categoricals).
if session_filter is not None:
    if isinstance(session_filter, tuple) and len(session_filter) == 2:
        s0, s1 = int(session_filter[0]), int(session_filter[1])
        if s0 > s1:
            s0, s1 = s1, s0
        long_df = long_df[long_df["session"].between(s0, s1)].copy()
        print(f"Filtered to sessions {s0}-{s1}: {len(long_df)} rows")
    else:
        keep_sessions = sorted({int(s) for s in session_filter})
        if len(keep_sessions) == 0:
            raise ValueError("session_filter list is empty")
        long_df = long_df[long_df["session"].isin(keep_sessions)].copy()
        print(f"Filtered to sessions {keep_sessions}: {len(long_df)} rows")

if long_df.empty:
    raise ValueError("No rows left after applying restriction/session filters")

# Set modeling categories and centered continuous predictors.
if restriction_filter == "both":
    long_df["restriction"] = pd.Categorical(long_df["restriction"], categories=["water restricted", "food restricted"])
else:
    long_df["restriction"] = pd.Categorical(long_df["restriction"], categories=[restriction_filter])
long_df["solution"] = pd.Categorical(
    long_df["solution"],
    categories=["dry", "water", "10% sucrose", "20% sucrose", "saccharine"],
)
_prev_cats = (
    ["dry", "water", "10% sucrose", "20% sucrose", "saccharine", "none"]
    if CATEGORICAL_CONTRAST == "sum"
    else ["none", "dry", "water", "10% sucrose", "20% sucrose", "saccharine"]
)
long_df["prev_solution"] = pd.Categorical(long_df["prev_solution"], categories=_prev_cats)

# Drop unused category levels to reduce rank-deficiency risks.
for c in ["restriction", "solution", "prev_solution"]:
    long_df[c] = long_df[c].cat.remove_unused_categories()

# Fixed-effect formula (omit C(restriction) when only one state is in the data).
USE_RESTRICTION_IN_MODEL = restriction_filter == "both"


def _C(col):
    """patsy contrast: Sum only for prev_solution when CATEGORICAL_CONTRAST == "sum"; else treatment."""
    if CATEGORICAL_CONTRAST == "sum" and col == "prev_solution":
        return f"C({col}, Sum)"
    return f"C({col})"


def _lick_x_suffix():
    parts = []
    if LICK_X_RESTRICTION and USE_RESTRICTION_IN_MODEL:
        parts.append(f"{_C('restriction')}:lick_count_c")
    if LICK_X_SOLUTION:
        parts.append(f"{_C('solution')}:lick_count_c")
    return (" + " + " + ".join(parts)) if parts else ""


def _restriction_x_solution_suffix():
    if RESTRICTION_X_SOLUTION and USE_RESTRICTION_IN_MODEL:
        return f" + {_C('restriction')}:{_C('solution')}"
    return ""


_bc = ["session_c", "trial_c"]
if USE_RESTRICTION_IN_MODEL:
    _bc.append(_C("restriction"))
_bc.extend([_C("solution"), _C("prev_solution"), "lick_count_c"])
BASE_CORE = " + ".join(_bc)

LICK_X_SUFFIX = _lick_x_suffix()
RXS_SUFFIX = _restriction_x_solution_suffix()
full_interaction = f"signal ~ {BASE_CORE}{LICK_X_SUFFIX}{RXS_SUFFIX}"

_rs = ["trial_c"]
if USE_RESTRICTION_IN_MODEL:
    _rs.append(_C("restriction"))
_rs.extend([_C("solution"), _C("prev_solution"), "lick_count_c"])
reduced_no_session = f"signal ~ {' + '.join(_rs)}{LICK_X_SUFFIX}{RXS_SUFFIX}"

_me = ["session_c", "trial_c"]
if USE_RESTRICTION_IN_MODEL:
    _me.append(_C("restriction"))
_me.extend([_C("solution"), _C("prev_solution"), "lick_count_c"])
if LICK_X_RESTRICTION and USE_RESTRICTION_IN_MODEL:
    _me.append(f"{_C('restriction')}:lick_count_c")
if LICK_X_SOLUTION:
    _me.append(f"{_C('solution')}:lick_count_c")
if RESTRICTION_X_SOLUTION and USE_RESTRICTION_IN_MODEL:
    _me.append(f"{_C('restriction')}:{_C('solution')}")
main_effects = "signal ~ " + " + ".join(_me)

print(
    "Model flags: CATEGORICAL_CONTRAST =",
    CATEGORICAL_CONTRAST,
    "| LICK_X_RESTRICTION =",
    LICK_X_RESTRICTION,
    "| LICK_X_SOLUTION =",
    LICK_X_SOLUTION,
    "| RESTRICTION_X_SOLUTION =",
    RESTRICTION_X_SOLUTION,
)
print("Full formula:", full_interaction)

long_df["session_c"] = long_df["session"] - long_df["session"].mean()
long_df["trial_c"] = long_df["trial"] - long_df["trial"].mean()
long_df["lick_count_c"] = long_df["lick_count"] - long_df["lick_count"].mean()

print("Unique mice:", long_df["mouse"].nunique())
print("Unique sessions (day labels):", long_df["session"].nunique())
print("Unique trial IDs (mouse x restriction x day x trial):", long_df[["mouse", "restriction", "session", "trial"]].drop_duplicates().shape[0])
print("Solution counts:\n", long_df["solution"].value_counts(dropna=False))
print("Prev solution counts:\n", long_df["prev_solution"].value_counts(dropna=False))

In [ ]:
print(long_df[["mouse", "restriction", "session", "trial"]].drop_duplicates().shape[0])
print(long_df["restriction"].value_counts(dropna=False))
print(pd.crosstab(long_df["restriction"], long_df["session"]))

In [ ]:
# Mixed-effects model with step-down fallbacks for singular fits.
# We first try the full model, then simplify if needed.

long_df["mouse_session"] = (
    long_df["mouse"].astype(str)
    + "__"
    + long_df["restriction"].astype(str)
    + "__"
    + long_df["session"].astype(str)
)

# full_interaction, reduced_no_session, main_effects are built in the categories cell (optional LICK_X_* / RESTRICTION_X_SOLUTION).

candidate_specs = [
    {
        "name": "full_vc",
        "formula": full_interaction,
        "vc_formula": {"mouse_session": "0 + C(mouse_session)"},
        "re_formula": "1",
    },
    {
        "name": "full_no_vc",
        "formula": full_interaction,
        "vc_formula": None,
        "re_formula": "1",
    },
    {
        "name": "reduced_no_session_no_vc",
        "formula": reduced_no_session,
        "vc_formula": None,
        "re_formula": "1",
    },
    {
        "name": "main_effects_only_no_vc",
        "formula": main_effects,
        "vc_formula": None,
        "re_formula": "1",
    },
]

fit = None
fit_spec_name = None
fit_formula = None

for spec in candidate_specs:
    print(f"\nTrying model spec: {spec['name']}")
    try:
        md = smf.mixedlm(
            formula=spec["formula"],
            data=long_df,
            groups=long_df["mouse"],
            vc_formula=spec["vc_formula"],
            re_formula=spec["re_formula"],
        )
    except Exception as e:
        print(f"  Failed to build model: {e}")
        continue

    for method in ["lbfgs", "bfgs", "cg", "powell"]:
        try:
            fit = md.fit(method=method, reml=False, maxiter=2000)
            fit_spec_name = spec["name"]
            fit_formula = spec["formula"]
            print(f"  Converged with optimizer: {method}")
            break
        except Exception as e:
            print(f"  Optimizer {method} failed: {e}")

    if fit is not None:
        break

if fit is None:
    raise RuntimeError("All candidate mixed-model specifications failed (singular matrix or convergence).")

print(f"Selected model spec: {fit_spec_name}")
print(f"Selected formula: {fit_formula}")
print(fit.summary())

In [ ]:
# Save coefficient table and model metadata.
coef = pd.DataFrame({
    "term": fit.params.index,
    "estimate": fit.params.values,
    "se": fit.bse.values,
    "z": fit.tvalues.values,
    "p": fit.pvalues.values,
})

ci = fit.conf_int()
coef["ci_low"] = ci[0].values
coef["ci_high"] = ci[1].values
coef["model_spec"] = fit_spec_name
coef["model_formula"] = fit_formula

# Per-1-SD scaling for main continuous fixed effects (same as forest plot).
_cont_std = {
    "session_c": float(long_df["session_c"].std(ddof=1)),
    "trial_c": float(long_df["trial_c"].std(ddof=1)),
    "lick_count_c": float(long_df["lick_count_c"].std(ddof=1)),
}
coef["predictor_sd"] = coef["term"].map(_cont_std)
coef["estimate_per_1sd"] = coef["estimate"] * coef["predictor_sd"]
coef["ci_low_per_1sd"] = coef["ci_low"] * coef["predictor_sd"]
coef["ci_high_per_1sd"] = coef["ci_high"] * coef["predictor_sd"]

coef_path = output_dir / "mixed_model_coefficients.csv"
coef.to_csv(coef_path, index=False)
print(f"Saved: {coef_path}")

# Save a readable text summary.
summary_path = output_dir / "mixed_model_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(f"Model spec: {fit_spec_name}\n")
    f.write(f"Formula: {fit_formula}\n\n")
    f.write(str(fit.summary()))
print(f"Saved: {summary_path}")

## Visualizing mixed-model results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", context="talk")

# Forest plot: key coefficients (estimate +/- 95% CI), same order as coefficient table

from matplotlib.lines import Line2D

alpha = 0.05
plot_mask = coef["term"].str.contains(
    r"session_c|trial_c|lick_count_c|C\(restriction|C\(solution|C\(prev_solution",
    regex=True,
    na=False,
)
plot_base = coef.loc[plot_mask, ["term", "estimate", "ci_low", "ci_high", "p"]].copy()

# Scale main continuous terms by SD(session_c), SD(trial_c), SD(lick_count_c) so estimates are
# "change in signal per 1 SD" on each centered predictor. Categorical terms unchanged.
_cont_std = {
    "session_c": float(long_df["session_c"].std(ddof=1)),
    "trial_c": float(long_df["trial_c"].std(ddof=1)),
    "lick_count_c": float(long_df["lick_count_c"].std(ddof=1)),
}
plot_forest = plot_base.copy()
for _t, _s in _cont_std.items():
    if _s > 0 and np.isfinite(_s):
        _m = plot_forest["term"] == _t
        for _col in ("estimate", "ci_low", "ci_high"):
            plot_forest.loc[_m, _col] = plot_forest.loc[_m, _col] * _s

def forest_plot(plot_df, title_suffix):
    """One row per term; y order matches row order in plot_df (top = first row)."""
    n = len(plot_df)
    fig_h = max(6, 0.35 * n)
    fig, ax = plt.subplots(figsize=(12, fig_h))
    y_pos = np.arange(n)
    color_sig = "#2171b5"
    color_ns = "#bdbdbd"
    for i in range(n):
        row = plot_df.iloc[i]
        c = color_sig if row["p"] < alpha else color_ns
        xerr = np.array([[row["estimate"] - row["ci_low"]], [row["ci_high"] - row["estimate"]]])
        ax.errorbar(
            row["estimate"],
            i,
            xerr=xerr,
            fmt="o",
            color=c,
            ecolor=c,
            capsize=3,
            markersize=6,
        )
    ax.axvline(0, linestyle="--", color="black", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["term"])
    ax.invert_yaxis()
    ax.set_xlabel(
        "Coefficient (95% CI): continuous = change in signal per 1 SD of predictor; "
        "categorical = contrast in raw signal units"
    )
    ax.set_ylabel("Term")
    ax.set_title(
        f"Model coefficients ({EPOCH}) — {title_suffix}\n"
        f"(blue: p < {alpha}; gray: not significant; p-values from unscaled fit)"
    )
    legend_elems = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color_sig, markersize=8, label=f"p < {alpha}"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color_ns, markersize=8, label=f"p ≥ {alpha}"),
    ]
    ax.legend(handles=legend_elems, loc="center right")
    plt.tight_layout()
    plt.show()

plot_table_order = plot_forest.copy()
forest_plot(plot_table_order, "same order as coefficient table")
print("SD scales (continuous, for forest plot):", {k: round(v, 4) for k, v in _cont_std.items()})


## Drop-one analysis (mixed-effects model)

Each row fits the **full fixed-effects model** with **one variable family removed** from the formula above. Compares log-likelihood and AIC to the **full** model (same random-effects structure as `fit_spec_name`). Larger **ΔAIC** and smaller **LRT p** mean dropping that variable hurts fit more.

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt

def _lick_x_suffix_drop(dropped=None):
    parts = []
    if LICK_X_RESTRICTION and USE_RESTRICTION_IN_MODEL:
        if dropped != "lick_x_restriction":
            parts.append(f"{_C('restriction')}:lick_count_c")
    if LICK_X_SOLUTION:
        if dropped != "lick_x_solution":
            parts.append(f"{_C('solution')}:lick_count_c")
    return (" + " + " + ".join(parts)) if parts else ""

def _drop_one_formula(dropped=None):
    """Full model with one predictor family removed; optional lick×(restriction|solution) terms."""
    ps = []
    if dropped != "session_c":
        ps.append("session_c")
    if dropped != "trial_c":
        ps.append("trial_c")
    if USE_RESTRICTION_IN_MODEL and dropped != "restriction":
        ps.append(_C("restriction"))
    if dropped != "solution":
        ps.append(_C("solution"))
    if dropped != "prev_solution":
        ps.append(_C("prev_solution"))
    if dropped != "lick_count_c":
        ps.append("lick_count_c")
    inner = " + ".join(ps)
    lx = _lick_x_suffix_drop(dropped)
    rxs = ""
    if RESTRICTION_X_SOLUTION and USE_RESTRICTION_IN_MODEL and dropped != "restriction_x_solution":
        rxs = f" + {_C('restriction')}:{_C('solution')}"
    return f"signal ~ {inner}{lx}{rxs}"

FULL_DROP_ONE = _drop_one_formula()

if USE_RESTRICTION_IN_MODEL:
    DROP_ONE_FORMULAS = {
        "session_c": _drop_one_formula("session_c"),
        "trial_c": _drop_one_formula("trial_c"),
        "lick_count_c": _drop_one_formula("lick_count_c"),
        "restriction": _drop_one_formula("restriction"),
        "solution": _drop_one_formula("solution"),
        "prev_solution": _drop_one_formula("prev_solution"),
    }
else:
    DROP_ONE_FORMULAS = {
        "session_c": _drop_one_formula("session_c"),
        "trial_c": _drop_one_formula("trial_c"),
        "lick_count_c": _drop_one_formula("lick_count_c"),
        "solution": _drop_one_formula("solution"),
        "prev_solution": _drop_one_formula("prev_solution"),
    }
if LICK_X_RESTRICTION and USE_RESTRICTION_IN_MODEL:
    DROP_ONE_FORMULAS["lick_x_restriction"] = _drop_one_formula("lick_x_restriction")
if LICK_X_SOLUTION:
    DROP_ONE_FORMULAS["lick_x_solution"] = _drop_one_formula("lick_x_solution")
if RESTRICTION_X_SOLUTION and USE_RESTRICTION_IN_MODEL:
    DROP_ONE_FORMULAS["restriction_x_solution"] = _drop_one_formula("restriction_x_solution")

# Match random-effects structure to the model that actually converged.
spec_lookup = {s["name"]: s for s in candidate_specs}
vc_spec = spec_lookup[fit_spec_name]["vc_formula"]
re_spec = spec_lookup[fit_spec_name]["re_formula"]


def fit_mixedlm_ml(formula_str):
    md = smf.mixedlm(
        formula=formula_str,
        data=long_df,
        groups=long_df["mouse"],
        vc_formula=vc_spec,
        re_formula=re_spec,
    )
    for method in ["lbfgs", "bfgs", "cg", "powell"]:
        try:
            return md.fit(method=method, reml=False, maxiter=2000)
        except Exception:
            continue
    return None

# Reference full model: reuse `fit` if it matches FULL_DROP_ONE; otherwise refit.
if fit_formula.replace(" ", "") == FULL_DROP_ONE.replace(" ", ""):
    fit_full_ref = fit
    print("Using existing `fit` as reference full model.")
else:
    print("Refitting full interaction model for drop-one reference (differs from selected fit_formula).")
    fit_full_ref = fit_mixedlm_ml(FULL_DROP_ONE)
    if fit_full_ref is None:
        raise RuntimeError("Could not fit FULL_DROP_ONE for drop-one analysis.")

ll_full = fit_full_ref.llf
aic_full = fit_full_ref.aic
k_full = len(fit_full_ref.params)

rows = []
for dropped, fml in DROP_ONE_FORMULAS.items():
    r = fit_mixedlm_ml(fml)
    if r is None:
        rows.append({
            "dropped": dropped,
            "converged": False,
            "n_params": np.nan,
            "ll": np.nan,
            "aic": np.nan,
            "delta_ll": np.nan,
            "delta_aic": np.nan,
            "lr_chi2": np.nan,
            "lr_df": np.nan,
            "lr_p": np.nan,
        })
        continue
    ll_r = r.llf
    k_r = len(r.params)
    lr_df = k_full - k_r
    lr_chi2 = 2.0 * (ll_full - ll_r) if lr_df > 0 else np.nan
    lr_p = chi2.sf(lr_chi2, lr_df) if lr_df > 0 and np.isfinite(lr_chi2) else np.nan
    rows.append({
        "dropped": dropped,
        "converged": True,
        "n_params": k_r,
        "ll": ll_r,
        "aic": r.aic,
        "delta_ll": ll_r - ll_full,
        "delta_aic": r.aic - aic_full,
        "lr_chi2": lr_chi2,
        "lr_df": lr_df,
        "lr_p": lr_p,
    })

drop_one_df = pd.DataFrame(rows)
# Sort by worst fit when removed (most positive delta AIC = larger penalty for dropping)
drop_one_df = drop_one_df.sort_values("delta_aic", ascending=False, na_position="last")

drop_path = output_dir / "drop_one_variable_comparison.csv"
drop_one_df.to_csv(drop_path, index=False)
print(f"Saved: {drop_path}")
print("\nReference full LL = {:.4f}, AIC = {:.2f}, k = {}".format(ll_full, aic_full, k_full))
print(drop_one_df.to_string(index=False))

# Bar plot: ΔAIC (positive = worse fit when variable removed)
plot_df = drop_one_df.dropna(subset=["delta_aic"]).sort_values("delta_aic", ascending=True)
if len(plot_df):
    plt.figure(figsize=(8, max(4, 0.35 * len(plot_df))))
    plt.barh(plot_df["dropped"], plot_df["delta_aic"], color="steelblue")
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("ΔAIC vs full (positive = worse without that variable)")
    plt.title("Drop-one variable importance (ΔAIC)")
    plt.tight_layout()
    plt.show()